# 01 · Exploring drives

Goal: see what the drive-level features look like and **which game-situation factors shift them**.
Anything that moves play-calling for reasons unrelated to *style* (score, field position, time)
must be controlled for before comparing drive styles' effectiveness (Phase 4).

Feature groups (see `drivedna.features.drive_features`):
- **STYLE**: how the offense chose to play. The only inputs to clustering.
- **RHYTHM**: how the drive unfolded (partly results). For describing, never for clustering.
- **CONTEXT / OUTCOME**: controls and targets, kept separate.

In [ ]:
import polars as pl
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from drivedna.drives.build import PROCESSED_DIR
from drivedna.features.drive_features import STYLE_FEATURES, RHYTHM_FEATURES
from drivedna.features.tokens import drive_token_sequences

feats = pl.read_parquet(PROCESSED_DIR / "drive_features.parquet")
plays = pl.read_parquet(PROCESSED_DIR / "drive_plays.parquet")
d = feats.filter(pl.col("keep"))
print(f"{feats.height:,} drives total, {d.height:,} kept after filters ({d.height / feats.height:.0%})")
print(d.group_by("season").len().sort("season").transpose(include_header=True).to_pandas().to_string(index=False, header=False))

## How drives end

In [ ]:
res = (d.group_by("result").agg(n=pl.len(), pts=pl.col("points").mean(), epa=pl.col("epa").mean())
         .with_columns(share=pl.col("n") / pl.col("n").sum()).sort("n", descending=True))
print(res)
px.bar(res.to_pandas(), x="result", y="share", title="Share of drives by result (kept drives)").show()

## Style feature distributions

In [ ]:
fig = make_subplots(rows=3, cols=4, subplot_titles=STYLE_FEATURES)
for i, c in enumerate(STYLE_FEATURES):
    fig.add_trace(go.Histogram(x=d[c].drop_nulls().to_list(), nbinsx=30, showlegend=False),
                  row=i // 4 + 1, col=i % 4 + 1)
fig.update_layout(height=700, title="Style features, one histogram each")
fig.show()
print(d.select(STYLE_FEATURES).null_count().transpose(include_header=True, column_names=["nulls"]).filter(pl.col("nulls") > 0))

**Missing values.** Depth features (`mean_air_yards`, `deep_pass_share`, `short_pass_share`) are null for
drives where no pass has a recorded depth, e.g. sack, sack, run. Sideline share is null without passes, and
inside-run share is null without runs. Phase 3 needs an explicit policy for these (impute plus an indicator, or restrict to drives with enough plays).

## Does game situation shift play-calling?
If so, comparing raw drive styles' success would confound style with game script.

In [ ]:
d2 = d.with_columns(
    score_bucket=pl.col("score_diff_start").cut([-15, -8, -1, 0, 7, 14], labels=["≤-15","-14..-8","-7..-1","0","1..7","8..14","≥15"], left_closed=False),
    qtr=pl.col("qtr_start").cast(pl.Int32),
)
def by(col, y, title):
    g = d2.group_by(col).agg(mean=pl.col(y).mean(), n=pl.len()).sort(col)
    fig = px.bar(g.to_pandas(), x=col, y="mean", hover_data=["n"], title=title)
    fig.show()
by("score_bucket", "pass_rate", "Pass rate by score differential at drive start")
by("score_bucket", "sec_per_play", "Seconds per play by score differential")
by("score_bucket", "deep_pass_share", "Deep-pass share by score differential")
by("qtr", "no_huddle_rate", "No-huddle rate by quarter")

In [ ]:
g = d2.group_by("start_yardline_100").agg(pass_rate=pl.col("pass_rate").mean(), pts=pl.col("points").mean(), n=pl.len()).filter(pl.col("n") > 100).sort("start_yardline_100")
px.line(g.to_pandas(), x="start_yardline_100", y="pts", title="Points per drive vs. start yardline (yards to opp end zone): field position drives outcomes by itself").show()

## How correlated are the style features?

In [ ]:
corr = d.select(STYLE_FEATURES).to_pandas().corr()
px.imshow(corr, zmin=-1, zmax=1, color_continuous_scale="RdBu_r", title="Style feature correlations", height=650).show()

## Play tokens: a drive as a sequence

In [ ]:
seq = drive_token_sequences(plays)
tok = seq.explode("tokens")
freq = tok.group_by("tokens").len().sort("len", descending=True).with_columns(share=pl.col("len") / pl.col("len").sum())
print(freq)
ex = (seq.join(feats.filter(pl.col("keep")).select("game_id","posteam","fixed_drive","result","season"), on=["game_id","posteam","fixed_drive"])
         .filter(pl.col("result") == "Touchdown").sample(8, seed=7))
for r in ex.iter_rows(named=True):
    print(f"{r['season']} {r['posteam']:>3} TD: {r['token_str']}")

## Takeaways to carry forward
- **Score matters for how much a team passes and how fast it plays.** Teams trailing by 8+ pass on ~68% of
  plays vs ~58% when leading, run ~3-4 s/play faster and go no-huddle ~10% vs ~6.5% of the time. Score
  differential must be a control in Phase 4.
- **Pass depth does not shift with score** (deep-pass share is ~20-23% in every bucket), so depth is a cleaner
  style signal than pass rate.
- **Field position is a huge outcome driver by itself:** ~4.6 points/drive starting near the opponent's end
  zone vs ~1.3 from your own goal line. Style comparisons must hold it fixed.
- **Redundancy:** pass rate, early-down pass rate and PROE are highly correlated (0.8-0.9), as are mean air yards
  and short-pass share (-0.7). Clustering will start with PCA rather than raw features.
- **Missingness** in depth/run-mix features needs a deliberate policy before Phase 3.
- Scrambles are flagged as dropbacks by nflverse, so `pass_rate` is a dropback rate (see `drive_features.py`).